In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q18/annotated/checkpoints/pre_cell_3.pickle

trying: ['lineitem']


me:  1
trying: ['load_orders']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['DATA_ROOT']
me:  0
trying: ['load_customer']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['customer']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  2


Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer


In [4]:
%%RecordEvent
%%time
### cell 3 ###

### cell 3 – optimized for cuDF

gb1 = lineitem.groupby(
    "L_ORDERKEY", as_index=False, sort=False
)["L_QUANTITY"].sum()
# filter orders whose total quantity > 300
# fixed the variable assignment here
gfb = gb1[gb1.L_QUANTITY > 300]
# join once with orders and once with customer, then sort
total = (
    gfb
    .merge(orders, left_on="L_ORDERKEY", right_on="O_ORDERKEY")
    .merge(customer, left_on="O_CUSTKEY", right_on="C_CUSTKEY")
    .sort_values(["O_TOTALPRICE", "O_ORDERDATE"], ascending=[False, True])
)

CPU times: user 55.4 ms, sys: 51.8 ms, total: 107 ms
Wall time: 107 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q18/rewritten/o4_mini_high/checkpoints/post_cell_3_try_3.pickle

migration speed (bps): 799652896.3976257
---------------------------
variables to migrate:
lineitem 48
load_lineitem 144
pd 72
customer 48
STORAGE_OPTS 64
orders 48
DATA_ROOT 80
load_orders 144
load_customer 144
gb1 48
total 48
gfb 48
tpch_parent 54
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q18/rewritten/o4_mini_high/checkpoints/post_cell_3_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['orders', 'lineitem']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['total', 'gfb', 'gb1']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q18/opt_cell_exec_info_3_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q18/annotated/checkpoints/post_cell_3.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
